# API Endpoint Access and Data Loading

In [1]:
# Load data and packages
import requests
import pandas as pd
import numpy as np 
import json
from urllib.request import urlopen
import streamlit as st

# Ensure project root is on sys.path to import Helpers package
import os
import sys
sys.path.append(os.path.abspath(".."))

from Helpers.utils import get_supabase

# Get sessions
base_url = "https://api.openf1.org/v1/sessions"
years = [2024, 2025]

# Helper function to get the rest of the data within a given session
def get_api_df(base_link: str, session_keys):
    all_dfs = []

    for key in session_keys:
        link = f"{base_link}?session_key={key}"
        response = requests.get(link)
        data = response.json()

        if not data:
            continue  # skip if no data

        if isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):
            df = pd.DataFrame(data)
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
        else:
            df = pd.DataFrame(data, index=[0])

        # keep track of which session each row came from if it's not already in data
        if 'session_key' not in df.columns:
            df['session_key'] = key

        all_dfs.append(df)

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        return pd.DataFrame()

# Fetch each year separately
all_sessions = []
for y in years:
    resp = requests.get(base_url, params={"year": y})
    resp.raise_for_status()
    all_sessions.extend(resp.json())

# Convert to DataFrame
sessions = pd.DataFrame(all_sessions)

# Filter again just in case and drop columns
sessions_races = sessions[sessions['year'].isin(years)].copy()
sessions_races.drop(columns=["gmt_offset", "country_code"], inplace=True)
sessions_races[sessions_races["year"] == 2025].head()



2025-10-27 11:11:57.134 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-27 11:11:57.393 
  command:

    streamlit run /Users/baole/Documents/F1 Machine Learning Model/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-10-27 11:11:57.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-27 11:11:57.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-27 11:11:57.395 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-27 11:11:57.633 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-27 11:11:57.634 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-1

,meeting_key,session_key,location,date_start,date_end,session_type,session_name,country_key,country_name,circuit_key,circuit_short_name,year
123,1253,9683,Sakhir,2025-02-26T07:00:00+00:00,2025-02-26T16:00:00+00:00,Practice,Day 1,36,Bahrain,63,Sakhir,2025
124,1253,9684,Sakhir,2025-02-27T07:00:00+00:00,2025-02-27T16:00:00+00:00,Practice,Day 2,36,Bahrain,63,Sakhir,2025
125,1253,9685,Sakhir,2025-02-28T07:00:00+00:00,2025-02-28T16:00:00+00:00,Practice,Day 3,36,Bahrain,63,Sakhir,2025
126,1254,9686,Melbourne,2025-03-14T01:30:00+00:00,2025-03-14T02:30:00+00:00,Practice,Practice 1,5,Australia,10,Melbourne,2025
127,1254,9687,Melbourne,2025-03-14T05:00:00+00:00,2025-03-14T06:00:00+00:00,Practice,Practice 2,5,Australia,10,Melbourne,2025


In [2]:
# Get unique session keys
session_keys = sessions_races["session_key"].unique()

In [3]:
# Get driver data
url = "https://api.openf1.org/v1/drivers"
driver_df = get_api_df(url,session_keys)

In [4]:
# Driver_df column alterations
driver_df.drop(columns=["broadcast_name", "full_name", "team_colour", "first_name","last_name","headshot_url", "detail", "error"], inplace=True, errors="ignore")
driver_df.replace(np.nan, None, inplace=True)
driver_df["meeting_key"] = pd.to_numeric(driver_df["meeting_key"], errors='coerce').astype('Int64')
driver_df["driver_number"] = pd.to_numeric(driver_df["driver_number"], errors='coerce').astype('Int64')  # Fixed: proper conversion
driver_df[driver_df["session_key"] > 9900]


,meeting_key,session_key,driver_number,name_acronym,team_name,country_code
2463,1255,9988,1,VER,Red Bull Racing,None
2464,1255,9988,4,NOR,McLaren,None
2465,1255,9988,5,BOR,Kick Sauber,None
2466,1255,9988,6,HAD,Racing Bulls,None
2467,1255,9988,7,DOO,Alpine,None
...,...,...,...,...,...,...
4038,1269,9904,44,HAM,Ferrari,None
4039,1269,9904,55,SAI,Williams,None
4040,1269,9904,63,RUS,Mercedes,None
4041,1269,9904,81,PIA,McLaren,None


In [5]:
# lap data
url = "https://api.openf1.org/v1/laps"
laps_df = get_api_df(url,session_keys)

In [6]:
# lap_df alterations
laps_df.replace(np.nan, None, inplace=True)
laps_df.drop(columns=["segments_sector_1","segments_sector_2","segments_sector_3","i1_speed","i2_speed"], inplace=True, errors="ignore")

In [7]:
# standings/results
url = "https://api.openf1.org/v1/session_result"
results_df = get_api_df(url,session_keys)

In [8]:
# results_df alterations
results_df.drop(columns=["detail","error"],inplace=True, errors="ignore")
results_df.replace(np.nan, None, inplace = True)

results_df.fillna(0, inplace=True)
columns_to_convert = ["position", "driver_number", "meeting_key"]
for i in columns_to_convert:
    results_df[i] = results_df[i].astype(int)
    
results_df.head()

/var/folders/59/0pv0md114td124t9vtdq3nhc0000gn/T/ipykernel_7702/1947560370.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  results_df.fillna(0, inplace=True)


,position,driver_number,number_of_laps,dnf,dns,dsq,duration,gap_to_leader,meeting_key,session_key,points
0,1,1,143.0,False,False,False,91.344,0.0,1228,9462,0.0
1,2,4,73.0,False,False,False,92.484,1.14,1228,9462,0.0
2,3,55,69.0,False,False,False,92.584,1.24,1228,9462,0.0
3,4,3,52.0,False,False,False,92.599,1.255,1228,9462,0.0
4,5,10,61.0,False,False,False,92.805,1.461,1228,9462,0.0


In [9]:
# pit data
url = "https://api.openf1.org/v1/pit"
pit_df = get_api_df(url,session_keys)

In [10]:
# pit_df alterations
pit_df.drop(columns=["detail","error"],inplace=True, errors="ignore")
pit_df.replace(np.nan, None, inplace = True)

pit_df.fillna(0, inplace=True)
columns_to_convert = ["lap_number", "driver_number", "meeting_key"]
for i in columns_to_convert:
    pit_df[i] = pit_df[i].astype(int)

   
pit_df = pit_df[pit_df["date"] != 0]
pit_df = pit_df.dropna(subset=["date"])

pit_df = pit_df.drop_duplicates(
    subset=['meeting_key', 'session_key', 'driver_number', 'lap_number'], 
    keep='last'
)
# Get the driver keys that were actually upserted to the database
# Use the same filtering logic that was applied during the drivers upsert
session_keys_window = set(zip(sessions_races['meeting_key'], sessions_races['session_key']))
driver_filt = driver_df.copy()

if {'meeting_key','session_key'}.issubset(driver_filt.columns):
    driver_filt = driver_filt[
        driver_filt.apply(lambda r: (r['meeting_key'], r['session_key']) in session_keys_window, axis=1)
    ]

# Also apply the PK cleaning that was done in the main process
driver_filt = driver_filt.dropna(subset=['meeting_key','session_key'])
for col in ['meeting_key','session_key']:
    driver_filt[col] = driver_filt[col].astype(int)

# Create valid driver keys from the filtered drivers
valid_driver_keys = set(
    zip(driver_filt['meeting_key'], driver_filt['session_key'], driver_filt['driver_number'])
)

print(f"Valid driver keys: {len(valid_driver_keys)}")

# Filter pit_df to only rows with valid driver keys
pit_df = pit_df[
    pit_df.apply(
        lambda row: (row['meeting_key'], row['session_key'], row['driver_number']) in valid_driver_keys, 
        axis=1
    )
]

print(f"Pit data after FK filtering: {len(pit_df)} rows")

pit_df

Valid driver keys: 4334
Pit data after FK filtering: 16046 rows


/var/folders/59/0pv0md114td124t9vtdq3nhc0000gn/T/ipykernel_7702/923753079.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pit_df.fillna(0, inplace=True)


,date,session_key,pit_duration,lap_number,driver_number,meeting_key
0,2024-02-21T07:00:04.303000+00:00,9462,0.0,1,63,1228
1,2024-02-21T07:00:19.465000+00:00,9462,0.0,1,14,1228
2,2024-02-21T07:00:24.249000+00:00,9462,0.0,1,23,1228
3,2024-02-21T07:00:30.814000+00:00,9462,0.0,1,77,1228
4,2024-02-21T07:00:41.497000+00:00,9462,0.0,1,31,1228
...,...,...,...,...,...,...
16871,2025-10-26T21:09:09.698000+00:00,9877,23.3,47,44,1272
16872,2025-10-26T21:10:22.169000+00:00,9877,22.1,48,87,1272
16873,2025-10-26T21:10:24.916000+00:00,9877,22.4,48,63,1272
16874,2025-10-26T21:11:28.773000+00:00,9877,22.7,48,43,1272


In [11]:
pk_cols = ['meeting_key', 'session_key', 'driver_number', 'lap_number']
duplicates = pit_df[pit_df.duplicated(subset=pk_cols, keep=False)]
print(f"Found {len(duplicates)} duplicate rows:")
if len(duplicates) > 0:
    print(duplicates[pk_cols].head(25))

Found 0 duplicate rows:


In [12]:
# weather f1 
url = "https://api.openf1.org/v1/weather"
weather_df = get_api_df(url,session_keys)

In [13]:
# Weather aggregation to daily granularity per meeting/session
# - one row per ['meeting_key','session_key','date'] where date is day-only
# - numeric columns averaged, non-numeric take the last observed value

# Drop unnecessary columns if present
weather_df = weather_df.drop(columns=["detail", "error"], errors="ignore")

# Replace np.nan with None so supabase won't choke on NaNs
weather_df = weather_df.replace({np.nan: None})

# --- Clean date column ---
# Coerce to datetime (invalid -> NaT), keep UTC, then drop invalid rows
weather_df['date'] = pd.to_datetime(weather_df['date'], errors='coerce', utc=True)
weather_df = weather_df.dropna(subset=['date'])
# Convert to date strings instead of date objects
weather_df['date'] = weather_df['date'].dt.strftime('%Y-%m-%d')

# --- Clean numeric columns ---
# Convert all number-like columns to numeric (invalid -> NaN)
numeric_cols_all = weather_df.select_dtypes(include=['object']).columns
for col in numeric_cols_all:
    # try to coerce to numeric if it looks number-like
    weather_df[col] = pd.to_numeric(weather_df[col], errors='ignore')

# Decide aggregations
numeric_cols = weather_df.select_dtypes(include=['number']).columns.tolist()
# Ensure keys aren't aggregated numerically - remove ALL grouping keys
group_keys = ['meeting_key', 'session_key', 'date']
for k in group_keys:
    if k in numeric_cols:
        numeric_cols.remove(k)

agg_map = {col: 'mean' for col in numeric_cols}

# For non-numeric, keep the last non-null value
non_numeric_cols = [c for c in weather_df.columns
                    if c not in numeric_cols and c not in group_keys]
for col in non_numeric_cols:
    agg_map[col] = 'last'

group_keys = ['meeting_key','session_key','date']
weather_daily = (weather_df
                 .groupby(group_keys, as_index=False)
                 .agg(agg_map))

# Normalize integer-like fields to integers where possible - CORRECTED VERSION
int_like = ['humidity','meeting_key','session_key']
for col in int_like:
    if col in weather_daily.columns:
        # First convert to numeric (coercing invalid values to NaN)
        numeric_series = pd.to_numeric(weather_daily[col], errors='coerce')
        
        # Only convert to Int64 if we have actual numeric values and they're integers or can be safely converted
        if not numeric_series.isna().all():
            # Round to handle any float values from aggregation, then convert to Int64
            try:
                # For averaged values, round them first
                rounded_series = numeric_series.round().astype('Int64')
                weather_daily[col] = rounded_series
            except (TypeError, ValueError) as e:
                print(f"Warning: Could not convert column '{col}' to Int64: {e}")
                # Keep as float if conversion fails
                weather_daily[col] = numeric_series
        else:
            # If all values are NaN after conversion, keep as object or handle appropriately
            print(f"Warning: Column '{col}' contains no valid numeric values")

# For upsert, make 'date' a string 'YYYY-MM-DD' (stable PK)
weather_daily['date'] = pd.to_datetime(weather_daily['date']).dt.strftime('%Y-%m-%d')

# Final dedupe guard (should already be unique)
weather_daily = (weather_daily
                 .sort_values(group_keys)
                 .drop_duplicates(subset=group_keys, keep='last'))

print(f"Weather daily rows: {len(weather_daily)}")
dupes = weather_daily[weather_daily.duplicated(subset=group_keys, keep=False)]
print(f"Remaining duplicates on {group_keys}: {len(dupes)}")

weather_df.head()

Weather daily rows: 216
Remaining duplicates on ['meeting_key', 'session_key', 'date']: 0


/var/folders/59/0pv0md114td124t9vtdq3nhc0000gn/T/ipykernel_7702/4054097952.py:23: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  weather_df[col] = pd.to_numeric(weather_df[col], errors='ignore')


,date,session_key,humidity,pressure,rainfall,track_temperature,wind_speed,meeting_key,wind_direction,air_temperature
0,2024-02-21,9462,61.0,1020.5,0.0,28.6,0.9,1228.0,109.0,21.8
1,2024-02-21,9462,61.0,1020.5,0.0,28.6,1.1,1228.0,1.0,21.8
2,2024-02-21,9462,60.0,1020.5,0.0,28.9,0.7,1228.0,48.0,21.9
3,2024-02-21,9462,59.0,1020.4,0.0,29.2,1.1,1228.0,9.0,22.0
4,2024-02-21,9462,58.0,1020.4,0.0,29.4,1.1,1228.0,55.0,22.1


In [14]:
# Ensure meeting_key is present and correct for all tables using sessions mapping
session_to_meeting = sessions_races.set_index("session_key")["meeting_key"].to_dict()

def ensure_meeting_key(df, session_col="session_key", meeting_col="meeting_key"):
    if df is None or df.empty:
        return df
    df = df.copy()
    # Add column if missing
    if meeting_col not in df.columns:
        df[meeting_col] = None
    # Only fill where null/empty
    mask = df[meeting_col].isna()
    if session_col in df.columns:
        df.loc[mask, meeting_col] = df.loc[mask, session_col].map(session_to_meeting)
    return df

# Apply to all downstream frames
driver_df  = ensure_meeting_key(driver_df)
laps_df    = ensure_meeting_key(laps_df)
pit_df     = ensure_meeting_key(pit_df)
results_df = ensure_meeting_key(results_df)
# For weather_daily, meeting_key already aggregated; still ensure fill
try:
    weather_daily = ensure_meeting_key(weather_daily)
except NameError:
    pass

# Diagnostics: how many null PK fields remain per table
def pk_nulls(df, cols):
    if df is None or df.empty: 
        return 0, 0
    m = df[cols].isna().any(axis=1)
    return len(df), int(m.sum())

print("drivers:", pk_nulls(driver_df,  ["meeting_key","session_key","driver_number"]))
print("laps:",    pk_nulls(laps_df,    ["meeting_key","session_key","driver_number","lap_number"]))
print("pits:",    pk_nulls(pit_df,     ["meeting_key","session_key","driver_number","lap_number"]))
print("results:", pk_nulls(results_df, ["meeting_key","session_key","driver_number"]))
try:
    print("weather_daily:", pk_nulls(weather_daily, ["meeting_key","session_key","date"]))
except NameError:
    pass

drivers: (4343, 9)
laps: (125999, 0)
pits: (16046, 0)
results: (4398, 0)
weather_daily: (216, 0)


In [15]:
from math import ceil

supabase = get_supabase()

def validate_no_null_pk(df, cols, table_name):
    """Validate that no null values exist in primary key columns."""
    if df is None or df.empty:
        return df
    
    null_mask = df[cols].isna().any(axis=1)
    null_count = null_mask.sum()
    
    if null_count > 0:
        print(f"⚠️  {table_name}: Found {null_count} rows with null PK values - removing them")
        print(f"   Null rows in columns: {cols}")
        # Show sample of problematic rows
        null_rows = df[null_mask][cols].head(5)
        print(f"   Sample null rows:\n{null_rows}")
        return df[~null_mask].copy()
    else:
        print(f"✅ {table_name}: No null PK values found ({len(df)} rows)")
        return df

def clean_int(df, cols):
    """Drop NaNs in PK columns and cast to int."""
    df = df.dropna(subset=cols)
    for c in cols:
        df[c] = df[c].astype(int)
    return df

def batch_upsert(df, table_name, conflict_cols, supabase, batch_size=500):
    """
    Upserts df into Supabase table_name in batches.
    conflict_cols: list of column names matching the table's PK or unique index.
    """
    records = df.to_dict(orient="records")
    n = len(records)
    if n == 0:
        print(f"No rows to upsert for '{table_name}'")
        return

    n_batches = (n // batch_size) + (1 if n % batch_size else 0)
    print(f"\nUpserting {n} rows into '{table_name}' ({n_batches} batches)")
    conflict_str = ",".join(conflict_cols)   # 👈 Supabase expects string
    for i in range(n_batches):
        start = i * batch_size
        end = min(start + batch_size, n)
        batch = records[start:end]
        print(f"  Batch {i+1}/{n_batches}: rows {start}–{end-1}")
        resp = supabase.table(table_name).upsert(
            batch,
            on_conflict=conflict_str  # 👈 pass as string
        ).execute()
        # optional debugging:
        # print(resp)

    print(f"Done upserting '{table_name}'.\n")


def verify_counts(df, table_name, supabase):
    """Print count of rows in DataFrame vs. Supabase table."""
    try:
        res = supabase.table(table_name).select("*", count="exact").execute()
        print(f"Verification: '{table_name}' DataFrame rows = {len(df)}, Supabase rows = {res.count}")
    except Exception as e:
        print(f"Could not verify '{table_name}': {e}")

# --- Clean PK columns with enhanced validation ---
print("=== CLEANING PRIMARY KEY COLUMNS ===")

sessions_races = validate_no_null_pk(sessions_races, ["meeting_key","session_key"], "sessions")
sessions_races = clean_int(sessions_races, ["meeting_key","session_key"])

driver_df = validate_no_null_pk(driver_df, ["meeting_key","session_key","driver_number"], "drivers")
driver_df = clean_int(driver_df, ["meeting_key","session_key","driver_number"])

# Additional safety check for driver_number specifically
if 'driver_number' in driver_df.columns:
    null_driver_nums = driver_df['driver_number'].isna().sum()
    if null_driver_nums > 0:
        print(f"⚠️  Additional null driver_number check: Found {null_driver_nums} null values - removing rows")
        driver_df = driver_df.dropna(subset=['driver_number'])
        print(f"   Drivers after null removal: {len(driver_df)} rows")

laps_df = validate_no_null_pk(laps_df, ["meeting_key","session_key","driver_number","lap_number"], "laps")
laps_df = clean_int(laps_df, ["meeting_key","session_key","driver_number","lap_number"])

results_df = validate_no_null_pk(results_df, ["meeting_key","session_key","driver_number"], "results")
results_df = clean_int(results_df, ["meeting_key","session_key","driver_number"])

pit_df = validate_no_null_pk(pit_df, ["meeting_key","session_key","driver_number","lap_number"], "pits")
pit_df = clean_int(pit_df, ["meeting_key","session_key","driver_number","lap_number"])

weather_df = validate_no_null_pk(weather_df, ["meeting_key","session_key"], "weather")
weather_df = clean_int(weather_df, ["meeting_key","session_key"])

# --- FIRST: Upsert parent tables ---
batch_upsert(sessions_races, "sessions",
             ["meeting_key","session_key"], supabase)

# Filter and upsert drivers
session_keys_window = set(zip(sessions_races['meeting_key'], sessions_races['session_key']))
driver_filt = driver_df.copy()

if {'meeting_key','session_key'}.issubset(driver_filt.columns):
    driver_filt = driver_filt[
        driver_filt.apply(lambda r: (r['meeting_key'], r['session_key']) in session_keys_window, axis=1)
    ]

# Final validation before upserting drivers
print(f"Final driver validation: {len(driver_filt)} rows")
null_check = driver_filt[["meeting_key","session_key","driver_number"]].isna().any(axis=1).sum()
if null_check > 0:
    print(f"⚠️  CRITICAL: {null_check} rows still have null PK values - removing them")
    driver_filt = driver_filt.dropna(subset=["meeting_key","session_key","driver_number"])
    print(f"   Final driver count: {len(driver_filt)} rows")

batch_upsert(driver_filt, "drivers",
             ["meeting_key","session_key","driver_number"], supabase)

# --- THEN: Filter and upsert child tables ---
# Create valid driver keys from the filtered drivers that were actually upserted
valid_driver_keys = set(
    zip(driver_filt['meeting_key'], driver_filt['session_key'], driver_filt['driver_number'])
)

# Filter results
print(f"Results data before FK filtering: {len(results_df)} rows")
results_filtered = results_df[
    results_df.apply(
        lambda row: (row['meeting_key'], row['session_key'], row['driver_number']) in valid_driver_keys, 
        axis=1
    )
]
print(f"Results data after FK filtering: {len(results_filtered)} rows")

# Filter pits
print(f"Pit data before FK filtering: {len(pit_df)} rows")
pit_filtered = pit_df[
    pit_df.apply(
        lambda row: (row['meeting_key'], row['session_key'], row['driver_number']) in valid_driver_keys, 
        axis=1
    )
]
# Remove duplicates and invalid dates
pit_filtered = pit_filtered.drop_duplicates(
    subset=['meeting_key', 'session_key', 'driver_number', 'lap_number'], 
    keep='last'
)
pit_filtered = pit_filtered[pit_filtered["date"] != 0]
pit_filtered = pit_filtered.dropna(subset=["date"])
print(f"Pit data after filtering and deduplication: {len(pit_filtered)} rows")

# Filter laps
print(f"Laps data before FK filtering: {len(laps_df)} rows")
laps_filtered = laps_df[
    laps_df.apply(
        lambda row: (row['meeting_key'], row['session_key'], row['driver_number']) in valid_driver_keys, 
        axis=1
    )
]
print(f"Laps data after FK filtering: {len(laps_filtered)} rows")

# Upsert child tables
batch_upsert(results_filtered, "results",
             ["meeting_key","session_key","driver_number"], supabase)

batch_upsert(laps_filtered, "laps",
             ["meeting_key","session_key","driver_number","lap_number"], supabase)

batch_upsert(pit_filtered, "pits",
             ["meeting_key","session_key","driver_number","lap_number"], supabase)

batch_upsert(weather_daily, "weather",  # Use weather_daily, not weather_df
             ["meeting_key","session_key","date"], supabase)

# --- Post-upsert verification ---
verify_counts(sessions_races, "sessions", supabase)
verify_counts(driver_filt, "drivers", supabase)
verify_counts(laps_filtered, "laps", supabase)
verify_counts(results_filtered, "results", supabase)
verify_counts(pit_filtered, "pits", supabase)
verify_counts(weather_daily, "weather", supabase)

=== CLEANING PRIMARY KEY COLUMNS ===
✅ sessions: No null PK values found (226 rows)
⚠️  drivers: Found 9 rows with null PK values - removing them
   Null rows in columns: ['meeting_key', 'session_key', 'driver_number']
   Sample null rows:
      meeting_key  session_key  driver_number
917          1237         9527           <NA>
1218         1240         9558           <NA>
1619         1245         9591           <NA>
1820         1247         9612           <NA>
2121         1250         9639           <NA>
✅ laps: No null PK values found (125999 rows)
✅ results: No null PK values found (4398 rows)
✅ pits: No null PK values found (16046 rows)
✅ weather: No null PK values found (23741 rows)

Upserting 226 rows into 'sessions' (1 batches)
  Batch 1/1: rows 0–225
Done upserting 'sessions'.

Final driver validation: 4334 rows

Upserting 4334 rows into 'drivers' (9 batches)
  Batch 1/9: rows 0–499
  Batch 2/9: rows 500–999
  Batch 3/9: rows 1000–1499
  Batch 4/9: rows 1500–1999
  Batch 5